# IBKR API notebook for fetching historical market data

#### Prerequisites: 
- ```pip install -r requirements.txt``` to install the necessary packages.

- Launch Trader Workstation (TWS) and enable ActiveX API (```File->Global Configuration->API->Settings``` then check ```Enable ActiveX and Socket Clients``` and uncheck ```Read-Only API```. Do not forget to apply the settings).

#### 1. Connection to IBKR API local gateway

```File->Global Configuration->API->Settings```
et cocher 
```Enable ActiveX and Socket Clients```
et décocher 
```Read-Only API```

In [ ]:
import logging
logging.basicConfig(level=logging.DEBUG, format='%(asctime)s - %(levelname)s - %(message)s')
import pandas as pd
from ib_async import *
util.startLoop()
global historical_data_interval, duration

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=14)
if ib.isConnected():
    print("✅ Connected to IBKR API")
else:
    print("❌Failed to connect to IBKR API")
util.logToConsole(logging.INFO)

2025-10-06 16:49:27,575 - INFO - Connecting to 127.0.0.1:7497 with clientId 14...
2025-10-06 16:49:27,576 - INFO - Connected
2025-10-06 16:49:27,578 - DEBUG - <<< 178,20251006 16:49:26 Central European Standard Time
2025-10-06 16:49:27,579 - DEBUG - >>> 71,2,14,
2025-10-06 16:49:27,579 - INFO - Logged on to server version 178
2025-10-06 16:49:27,579 - DEBUG - <<< 15,1,DU027
2025-10-06 16:49:27,621 - DEBUG - <<< 9,1,1
2025-10-06 16:49:27,622 - DEBUG - <<< 4,2,-1,2104,Market data farm connection is OK:usfuture,
2025-10-06 16:49:27,622 - INFO - Warning 2104, reqId -1: Market data farm connection is OK:usfuture
2025-10-06 16:49:27,622 - DEBUG - <<< 4,2,-1,2104,Market data farm connection is OK:eufarm,
2025-10-06 16:49:27,623 - INFO - Warning 2104, reqId -1: Market data farm connection is OK:eufarm
2025-10-06 16:49:27,623 - DEBUG - <<< 4,2,-1,2104,Market data farm connection is OK:cashfarm,
2025-10-06 16:49:27,623 - INFO - Warning 2104, reqId -1: Market data farm connection is OK:cashfarm
2

✅ Connected to IBKR API


2025-10-06 21:17:12,194 - ERROR - Error 200, reqId 7: No security definition has been found for the request, contract: Forex('JPYUSD', exchange='IDEALPRO')
2025-10-06 21:17:16,643 - ERROR - Error 200, reqId 8: No security definition has been found for the request, contract: Forex('JPYUSD', exchange='IDEALPRO')


## Request Historical data

#### 2. Choose your contract

### List of contract to fetch : 

**Forex**:
- EURUSD : `Forex(pair='EURUSD', exchange='IDEALPRO')`
- GBPUSD : `Forex(pair='GBPUSD', exchange='IDEALPRO')`
- USDJPY : `Forex(pair='USDJPY', exchange='IDEALPRO')` 
- CHFUSD : `Forex(pair='CHFUSD', exchange='IDEALPRO')`
- AUDUSD : `Forex(pair='AUDUSD',
- 
**Stocks**:
- Apple Inc. : `Stock(symbol='AAPL', exchange='SMART', currency='USD')`

**Indices**:
- Nasdaq 100 : `Index('NDX', 'NASDAQ', 'USD')`
- S&P 500 : `Index('SPX', 'CBOE', 'USD')`
- Dow Jones : `Index('DJI', )`
- CAC40 : `Index('CAC40', 'MONEP', 'EUR')`
- DAX : `Index('DAX', 'EUREX', 'EUR')`
- NIKAI : `Index('NIKAI

**Futures**:
- Gold Futures December 2025 : `CFD(symbol='XAUUSD', exchange='SMART', currency='USD', lastTradeDateOrContractMonth='202512')`
- Petrol brent : `CFD(symbol='UKOIL', )
- Petrol WTI : `CFD(symbol='USOIL', )
- Argent : `CFD(symbol='XAGUSD' )
- Gaz naturel : `CFD(symbol='NGUSD')

**Cryptos**:
- Bitcoin : `Crypto('BTC', 'USD', 'PAXOS')`
- Ethereum : `Crypto('ETH', 'USD', 'PAXOS')`
- Ripple : `Crypto('XRP', 'USD', 'PAXOS')`
- Litecoin : `Crypto('LTC', 'USD', 'PAXOS')`


In [ ]:
# Define the contract HERE

# For futures, you need to specify either expiry or localSymbol
contract = Forex(pair='USDJPY', exchange='IDEALPRO')
 

#======================================================================
# Below => just some printing on contract chosen:
contract_details = ib.reqContractDetails(contract)
# Extract and display the desired fields from contract_details
filtered_details = [
    {
        "secType": detail.contract.secType,
        "conId": detail.contract.conId,
        "symbol": detail.contract.symbol,
        "exchange": detail.contract.exchange,
        "longName": detail.longName,
        "timezoneId": detail.timeZoneId,
        "tradingHours": "\n".join(
            [f"  {segment}" for segment in detail.tradingHours.split(";")]
        ),
        "liquidHours": "\n".join(
            [f"  {segment}" for segment in detail.liquidHours.split(";")]
        ),
        "minSize": detail.minSize,
    }
    for detail in contract_details
]

# Print the filtered details in a clear format
for idx, detail in enumerate(filtered_details, start=1):
    print(f"Contract Detail {idx}:")
    for key, value in detail.items():
        print(f"  {key}: {value}")
        print()

Contract chosen: Forex('USDJPY', exchange='IDEALPRO')
Contract Detail 1:
  secType: CASH

  conId: 15016059

  symbol: USD

  exchange: IDEALPRO

  longName: United States dollar

  timezoneId: US/Eastern

  tradingHours:   20251005:1715-20251006:1700
  20251006:1715-20251007:1700
  20251007:1715-20251008:1700
  20251008:1715-20251009:1700
  20251009:1715-20251010:1700

  liquidHours:   20251005:1715-20251006:1700
  20251006:1715-20251007:1700
  20251007:1715-20251008:1700
  20251008:1715-20251009:1700
  20251009:1715-20251010:1700

  minSize: 1.0



#### (Optional) Check first data timestamp available

In [20]:
timestamp = ib.reqHeadTimeStamp(contract, whatToShow='ASK', useRTH=False) # ASK for Forex
formatted_time = timestamp.strftime('%B %d, %Y, %H:%M')
logging.info(f"First date of data available: {formatted_time}")
logging.info(f"Timestamp: {timestamp}")

2025-10-06 15:44:45,628 - INFO - First date of data available: March 09, 2005, 04:30
2025-10-06 15:44:45,628 - INFO - Timestamp: 2005-03-09 04:30:00


#### 3. End date choice for data request

In [4]:
# yesterday's date - UTC format for IBKR API
end_date = (pd.Timestamp.now(tz='UTC') - pd.DateOffset(days=1)).strftime('%Y%m%d-%H:%M:%S')

In [14]:
# today's date - UTC format for IBKR API  
end_date = pd.Timestamp.now(tz='UTC').strftime('%Y%m%d-%H:%M:%S')

In [15]:
# custom end date - UTC format for IBKR API
# format: yyyymmdd-hh:mm:ss (UTC time, no timezone suffix needed)
end_date = '20230323-22:00:00'

#### 4. Main loop for fetching historical data

Args
- **contract**: Contract of interest.  
- **endDateTime**:  
    - Can be set to `''` to indicate the current time.  
    - Can be given as a `datetime.date` or `datetime.datetime`.  
    - Can be given as a string in `'yyyyMMdd HH:mm:ss'` format.  
    - If no timezone is given, the TWS login timezone is used.  
- **durationStr**: Time span of all the bars. Examples:  
    - `'60 S'`, `'30 D'`, `'13 W'`, `'6 M'`, `'10 Y'`, etc...
- **barSizeSetting**: Time period of one bar. Must be one of:  
    - `'1 secs'`, `'5 secs'`, `'10 secs'`, `'15 secs'`, `'30 secs'`,  
    - `'1 min'`, `'2 mins'`, `'3 mins'`, `'5 mins'`, `'10 mins'`, `'15 mins'`,  
    - `'20 mins'`, `'30 mins'`,  
    - `'1 hour'`, `'2 hours'`, `'3 hours'`, `'4 hours'`, `'8 hours'`,  
    - `'1 day'`, `'1 week'`, `'1 month'`.  
- **whatToShow**: Specifies the source for constructing bars. Must be one of:  
    - `'TRADES'`, `'MIDPOINT'`, `'BID'`, `'ASK'`, `'BID_ASK'`,  
    - `'ADJUSTED_LAST'`, `'HISTORICAL_VOLATILITY'`, `'OPTION_IMPLIED_VOLATILITY'`,  
    - `'REBATE_RATE'`, `'FEE_RATE'`, `'YIELD_BID'`, `'YIELD_ASK'`, `'YIELD_BID_ASK'`, `'YIELD_LAST'`.  
    - For `'SCHEDULE'`, use `:meth:.reqHistoricalSchedule`.  
- **useRTH**:  
    - If `True`, only show data from within Regular Trading Hours.  
    - If `False`, show all data.  
- **formatDate**:  
    - For an intraday request, setting to `2` will cause the returned date fields to be timezone-aware `datetime.datetime` with UTC timezone, instead of local timezone as used by TWS.  
- **keepUpToDate**:  
    - If `True`, a realtime subscription is started to keep the bars updated.  
    - `endDateTime` must be set empty (`''`) then.  
- **chartOptions**: Unknown.  
- **timeout**:  
    - Timeout in seconds after which to cancel the request and return an empty bar series. 
    - If the data request is huge, this parameter could spoil the request  
    - Set to `0` to wait indefinitely.  

In [9]:
historical_data_interval = '10 secs' # Candle period to fetch
request_duration = '20 Y'  # Duration in years (use Y, not "year"). Use a very big value if you want the maximum historical data, it will fetch the maximum available automatically.
price_source = 'ASK'  # 'BID', 'ASK', or 'TRADES' (note that for some symbols, (e.g. EURUSD) only 'BID' and 'ASK' are available)

bars = ib.reqHistoricalData(
        contract,
        endDateTime=end_date,
        durationStr=str(request_duration),
        barSizeSetting=str(historical_data_interval),
        whatToShow=price_source,
        useRTH=False, 
        formatDate=2,
        timeout = 0)

#### 5. Convert the list of bars to a data frame, print the first / last rows and remove useless columns:

In [10]:
bars[0]
new_df = util.df(bars)

display(new_df.head())
display(new_df.tail())
# Remove the 'volume', 'average', and 'barCount' columns from the DataFrame
# new_df = new_df.drop(columns=['volume', 'average', 'barCount'])
new_df = new_df.drop(columns=['average', 'barCount'])

# Display the updated DataFrame
new_df.head()

,date,open,high,low,close,volume,average,barCount
0,2025-01-26 22:15:00+00:00,1.10400,1.10470,1.10390,1.10455,-1.0,-1.0,-1
1,2025-01-26 22:15:10+00:00,1.10455,1.10455,1.10445,1.10445,-1.0,-1.0,-1
2,2025-01-26 22:15:20+00:00,1.10445,1.10445,1.10445,1.10445,-1.0,-1.0,-1
3,2025-01-26 22:15:30+00:00,1.10445,1.10460,1.10445,1.10460,-1.0,-1.0,-1
4,2025-01-26 22:15:40+00:00,1.10460,1.10460,1.10450,1.10455,-1.0,-1.0,-1


,date,open,high,low,close,volume,average,barCount
1537625,2025-10-03 20:59:10+00:00,1.25824,1.25850,1.25824,1.25833,-1.0,-1.0,-1
1537626,2025-10-03 20:59:20+00:00,1.25833,1.25849,1.25823,1.25832,-1.0,-1.0,-1
1537627,2025-10-03 20:59:30+00:00,1.25832,1.25851,1.25832,1.25851,-1.0,-1.0,-1
1537628,2025-10-03 20:59:40+00:00,1.25851,1.25851,1.25827,1.25843,-1.0,-1.0,-1
1537629,2025-10-03 20:59:50+00:00,1.25843,1.25845,1.25834,1.25841,-1.0,-1.0,-1


,date,open,high,low,close,volume
0,2025-01-26 22:15:00+00:00,1.10400,1.10470,1.10390,1.10455,-1.0
1,2025-01-26 22:15:10+00:00,1.10455,1.10455,1.10445,1.10445,-1.0
2,2025-01-26 22:15:20+00:00,1.10445,1.10445,1.10445,1.10445,-1.0
3,2025-01-26 22:15:30+00:00,1.10445,1.10460,1.10445,1.10460,-1.0
4,2025-01-26 22:15:40+00:00,1.10460,1.10460,1.10450,1.10455,-1.0


#### 5B. Save new DataFrame to CSV (only for no existing file)

**IMPORTANT**: Use the correct format file : `../marketData/{contract.symbol}_{candle_period}_{first_date}_to_{last_date}_{price_source}.csv`

Date format to use: `YYYYMMDD`

In [11]:
save_path = f"../marketData/{contract.symbol}_{historical_data_interval.replace(' ', '_')}_20250129_to_20251003_{price_source}.csv"

# Save the newly downloaded DataFrame
new_df.to_csv(save_path, index=True)
print(f"New data saved to: {save_path}")

# Optional verification
import pandas as pd
df_check = pd.read_csv(save_path, index_col=0)
display(df_check.head())
display(df_check.tail())

New data saved to: ../marketData/CHF_10_secs_20250129_to_20251003_ASK.csv


,date,open,high,low,close,volume
0,2025-01-26 22:15:00+00:00,1.10400,1.10470,1.10390,1.10455,-1.0
1,2025-01-26 22:15:10+00:00,1.10455,1.10455,1.10445,1.10445,-1.0
2,2025-01-26 22:15:20+00:00,1.10445,1.10445,1.10445,1.10445,-1.0
3,2025-01-26 22:15:30+00:00,1.10445,1.10460,1.10445,1.10460,-1.0
4,2025-01-26 22:15:40+00:00,1.10460,1.10460,1.10450,1.10455,-1.0


,date,open,high,low,close,volume
1537625,2025-10-03 20:59:10+00:00,1.25824,1.25850,1.25824,1.25833,-1.0
1537626,2025-10-03 20:59:20+00:00,1.25833,1.25849,1.25823,1.25832,-1.0
1537627,2025-10-03 20:59:30+00:00,1.25832,1.25851,1.25832,1.25851,-1.0
1537628,2025-10-03 20:59:40+00:00,1.25851,1.25851,1.25827,1.25843,-1.0
1537629,2025-10-03 20:59:50+00:00,1.25843,1.25845,1.25834,1.25841,-1.0


At this step, you have a **pandas DataFrame** containing the historical data for your chosen contract. You can use this DataFrame for further analysis or visualization as needed. 
In the next cell, you can update an existing csv file by merging it with the new data. 

## Additional features

#### DataFrame update (only for existing data)
Update the dataframe by merging the new data with old ones

**IMPORTANT**: Use the correct format file : `../marketData/{contract.symbol}_{candle_period}_{first_date}_to_{last_date}_{price_source}.csv`

Date format to use: `YYYYMMDD`

In [13]:
import pandas as pd
from Helpers import merge_ohlc_dataframes
import os

# Load your existing data - use index_col=0 to treat first column as index
existing_file_path = "../marketData/XAUUSD_10secs_20250126_to_20250828_ASK.csv" # Here, enter the correct file path fo the existing csv data file
existing_df = pd.read_csv(existing_file_path, index_col=0)
display(existing_df.head())

# Merge the dataframes
merged_df = merge_ohlc_dataframes(existing_df, new_df, frequency='10s') # Adjust frequency as needed
display(merged_df.head())
display(merged_df.tail())

# Save the merged dataframe
save_path = f"../marketData/{contract.symbol}_10secs_20250126_to_20250310_{price_source}.csv" # Here, enter the correct file path for the new csv data file
merged_df.to_csv(save_path, index=True)
print(f"Merged data saved to: {save_path}")
# Delete the original file if needed
if os.path.exists(existing_file_path):
    os.remove(existing_file_path)
    print(f"Deleted original file: {existing_file_path}")


,date,open,high,low,close
0,2025-01-26 23:00:00+00:00,2780.00,2780.00,2771.68,2771.88
1,2025-01-26 23:00:10+00:00,2771.88,2772.35,2769.87,2769.93
2,2025-01-26 23:00:20+00:00,2769.93,2773.09,2766.91,2767.91
3,2025-01-26 23:00:30+00:00,2767.91,2769.16,2767.88,2768.56
4,2025-01-26 23:00:40+00:00,2768.56,2770.39,2768.48,2768.98


First few missing timestamps: [Timestamp('2025-01-27 22:00:00+0000', tz='UTC'), Timestamp('2025-01-27 22:00:10+0000', tz='UTC'), Timestamp('2025-01-27 22:00:20+0000', tz='UTC'), Timestamp('2025-01-27 22:00:30+0000', tz='UTC'), Timestamp('2025-01-27 22:00:40+0000', tz='UTC')]


,date,open,high,low,close,volume
0,2025-01-26 23:00:00+00:00,2780.00,2780.00,2771.68,2771.88,NaN
1,2025-01-26 23:00:10+00:00,2771.88,2772.35,2769.87,2769.93,NaN
2,2025-01-26 23:00:20+00:00,2769.93,2773.09,2766.91,2767.91,NaN
3,2025-01-26 23:00:30+00:00,2767.91,2769.16,2767.88,2768.56,NaN
4,2025-01-26 23:00:40+00:00,2768.56,2770.39,2768.48,2768.98,NaN


,date,open,high,low,close,volume
1477975,2025-10-03 20:59:10+00:00,3887.86,3888.00,3886.62,3886.62,-1.0
1477976,2025-10-03 20:59:20+00:00,3886.62,3886.63,3886.08,3886.08,-1.0
1477977,2025-10-03 20:59:30+00:00,3886.08,3886.09,3886.08,3886.08,-1.0
1477978,2025-10-03 20:59:40+00:00,3886.08,3887.77,3885.58,3885.58,-1.0
1477979,2025-10-03 20:59:50+00:00,3885.58,3888.00,3885.58,3888.00,-1.0


Merged data saved to: ../marketData/XAUUSD_10secs_20250126_to_20250310_ASK.csv
Deleted original file: ../marketData/XAUUSD_10secs_20250126_to_20250828_ASK.csv


#### Checking data integrity

In [ ]:
from Helpers import checkDataFile, visualize_data_gaps
import matplotlib.pyplot as plt

symbol = 'GC'
interval = '10secs'
start_date = '20230321'
end_date = '20250829'
price_source= 'TRADES'

# 1. Load the existing dataframe
save_path = f"../marketData/{symbol}_{interval}_{start_date}_to_{end_date}_{price_source}.csv"

# Analyze data gaps
report = checkDataFile(
    file_path=save_path, 
    interval=interval
)

# Print summary
print(f"Analyzed {report['total_trading_days']} trading days")
print(f"Found {report['days_with_gaps']} days with gaps ({report['analysis_summary']['gap_percentage']:.2f}%)")
print(f"Total gaps detected: {report['total_gaps']}")

# Visualize the gaps
fig = visualize_data_gaps(report)
plt.show()

# To examine specific days with large gaps
problem_days = {date: data for date, data in report["gaps_by_date"].items() 
                if data["missing_points"] > 10}
print(f"Days with more than 10 missing points: {len(problem_days)}")
for date, data in sorted(problem_days.items()):
    print(f"{date}: Missing {data['missing_points']} of {data['expected_points']} points")